## Import

In [1]:
import os
import time
import numpy as np
import torch
import pandas as pd

device = 'cuda:4'

%load_ext autoreload
%autoreload 2

## Dataset

The data is the latent presentation learned using exp_protein_autoencoder.ipynb



In [2]:
d = 64

Zx = np.load(f'results/proteins/Zx_LR_{d//2}.npy')
Zy = np.load(f'results/proteins/Zy_LR_{d//2}.npy')

X, Y = torch.tensor(Zx).float().to(device), torch.tensor(Zy).float().to(device)

X, Y  = (X-X.mean(dim=0, keepdim=True))/X.std(dim=0, keepdim=True),  (Y-Y.mean(dim=0, keepdim=True))/Y.std(dim=0, keepdim=True), 

print(X.size(), Y.size())

torch.Size([4496, 32]) torch.Size([4496, 32])


## MI estimate

In [3]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        self.max_iteration = 1000
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

In [8]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(None, None, None, hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 1.5005271434783936 loss val= 1.4931533336639404 best val loss= 1.4931533336639404 best t= 0
finished: t= 51 loss= 0.4833582043647766 loss val= 0.528714656829834 best val loss= 0.5181342363357544 best t= 47
finished: t= 102 loss= 0.624242901802063 loss val= 0.4831831455230713 best val loss= 0.4754144549369812 best t= 92
finished: t= 153 loss= 0.4849129915237427 loss val= 0.4827609062194824 best val loss= 0.4433654844760895 best t= 152
finished: t= 204 loss= 0.5700376629829407 loss val= 0.673133909702301 best val loss= 0.4433654844760895 best t= 152
finished: t= 255 loss= 0.5542837977409363 loss val= 0.5074257254600525 best val loss= 0.43738657236099243 best t= 251
finished: t= 306 loss= 0.4242400527000427 loss val= 0.622502326965332 best val loss= 0.43591398000717163 best t= 293
finished: t= 357 loss= 0.4084605574607849 loss val= 0.48652753233909607 best val loss= 0.43591398000717163 best t= 293
finished: t= 408 loss= 0.5512478351593018 loss val= 0.5002081990242004 

In [9]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)


print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.015022367238998413 loss val= 0.015747860074043274 best val loss= 0.015747860074043274 best t= 0
finished: t= 51 loss= -2.673241138458252 loss val= 0.33718442916870117 best val loss= -0.32618600130081177 best t= 2
finished: t= 102 loss= -2.6542766094207764 loss val= 1.1791467666625977 best val loss= -0.32618600130081177 best t= 2
finished: t= 153 loss= -2.935945510864258 loss val= 1.9030308723449707 best val loss= -0.32618600130081177 best t= 2


est MI: 0.9356924295425415


In [11]:
## Vector copula estimation
from estimators.VCE import VCE

hyperparams.nde_type = 'FM'

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('est MI:', estimator.MI(X, Y))

K components= 5 copula transform= True
learning marginals...
nde type: FM
finished: t= 0 loss= 1.9678822755813599 loss val= 2.232966184616089 best val loss= 2.232966184616089 best t= 0
finished: t= 126 loss= 0.8824699521064758 loss val= 1.0960180759429932 best val loss= 1.0960180759429932 best t= 126
finished: t= 252 loss= 0.8220130205154419 loss val= 1.0865483283996582 best val loss= 1.0306987762451172 best t= 202
finished: t= 378 loss= 0.8637341856956482 loss val= 1.0355039834976196 best val loss= 0.9832038283348083 best t= 327
finished: t= 504 loss= 0.7898386716842651 loss val= 1.0828499794006348 best val loss= 0.9832038283348083 best t= 327


finished: t= 0 loss= 2.0603699684143066 loss val= 1.7089896202087402 best val loss= 1.7089896202087402 best t= 0
finished: t= 126 loss= 0.8189821839332581 loss val= 0.8876980543136597 best val loss= 0.8318493962287903 best t= 117
finished: t= 252 loss= 0.738815426826477 loss val= 0.8624889254570007 best val loss= 0.7959953546524048 best t= 227

## Self-consistency test

In [19]:
from self_consistency_test import *


shuffle_test_vce = shuffle_test(X, Y, VCE, device=device, name='sc_shuffle_test_vce')
print('shuffle_test=', shuffle_test_vce)

shuffle test ideal value: 0
K components= 5 copula transform= True
learning marginals...
nde type: FM
finished: t= 0 loss= 2.080965280532837 loss val= 2.074007511138916 best val loss= 2.074007511138916 best t= 0
finished: t= 126 loss= 0.8836250901222229 loss val= 1.0123990774154663 best val loss= 0.9563586115837097 best t= 122
finished: t= 252 loss= 0.9716191291809082 loss val= 0.9497230648994446 best val loss= 0.8869038820266724 best t= 245
finished: t= 378 loss= 0.8819491863250732 loss val= 0.934468150138855 best val loss= 0.8745779395103455 best t= 377
finished: t= 504 loss= 0.7975945472717285 loss val= 0.849425196647644 best val loss= 0.8449907898902893 best t= 443
finished: t= 630 loss= 0.7777073979377747 loss val= 0.8655573129653931 best val loss= 0.8265911340713501 best t= 541
finished: t= 756 loss= 0.8656103014945984 loss val= 0.8598246574401855 best val loss= 0.8176071047782898 best t= 705
finished: t= 882 loss= 0.7707290649414062 loss val= 0.9214498400688171 best val loss= 0.

In [ ]:
from self_consistency_test import *


dpi_test_vce = dpi_test(X, Y, VCE, p_array=[0.5], device=device, name='sc_dpi_test_vce')
print('dpi_test=', dpi_test_vce)

X is masked with ratio: 0.5
K components= 5 copula transform= True
learning marginals...
nde type: FM
finished: t= 0 loss= 1.973829746246338 loss val= 2.1806044578552246 best val loss= 2.1806044578552246 best t= 0
finished: t= 126 loss= 1.1646524667739868 loss val= 1.2651698589324951 best val loss= 1.2037047147750854 best t= 123
finished: t= 252 loss= 1.0128402709960938 loss val= 1.1718552112579346 best val loss= 1.1460164785385132 best t= 237
finished: t= 378 loss= 0.9968513250350952 loss val= 1.113913893699646 best val loss= 1.0773215293884277 best t= 356
finished: t= 504 loss= 0.9501450061798096 loss val= 1.1049023866653442 best val loss= 1.0755666494369507 best t= 463
finished: t= 630 loss= 0.9192764163017273 loss val= 1.1500111818313599 best val loss= 1.0755666494369507 best t= 463


finished: t= 0 loss= 2.039133071899414 loss val= 1.83362877368927 best val loss= 1.83362877368927 best t= 0
finished: t= 126 loss= 1.459883213043213 loss val= 1.420915961265564 best val loss= 1.406502

In [17]:
from self_consistency_test import *


additive_test_vce = additive_test(X, Y, VCE, device=device, name='sc_additive_test_vce')
print(additive_test_vce)

additive test ideal value: 2*I(X; Y)
K components= 5 copula transform= True
learning marginals...
nde type: FM
finished: t= 0 loss= 2.050326108932495 loss val= 2.1075046062469482 best val loss= 2.1075046062469482 best t= 0
finished: t= 126 loss= 1.0519598722457886 loss val= 1.222363829612732 best val loss= 1.1888643503189087 best t= 106
finished: t= 252 loss= 0.9971525073051453 loss val= 1.1437084674835205 best val loss= 1.1310715675354004 best t= 224
finished: t= 378 loss= 0.9222607016563416 loss val= 1.1355202198028564 best val loss= 1.1139557361602783 best t= 353
finished: t= 504 loss= 0.8994133472442627 loss val= 1.1400630474090576 best val loss= 1.101887583732605 best t= 474
finished: t= 630 loss= 0.9155552983283997 loss val= 1.1309736967086792 best val loss= 1.098508358001709 best t= 506
finished: t= 756 loss= 0.8686105012893677 loss val= 1.1185969114303589 best val loss= 1.093966007232666 best t= 671
finished: t= 882 loss= 0.8571896553039551 loss val= 1.1531985998153687 best val